 # Table of Contents
+ [Import](#Import_0)
	+ [Own function](#Own_function_1)
+ [Input and Output files](#Input_and_Output_files_2)
	+ [Input](#Input_3)
	+ [Output](#Output_4)
+ [Extract product](#Extract_product_5)
	+ [From title](#From_title_6)
	+ [From abstract](#From_abstract_7)
	+ [From full text](#From_full_text_8)
+ [Result](#Result_9)
	+ [Statistics ](#Statistics_10)
	+ [Format output](#Format_output_11)
+ [Summary](#Summary_12)


<a class="anchor" id="Import_0"></a>
# <span class=title_0 style="color: #1f9e89">Import</span>

In [1]:
# Standard library imports
import os
import sys
from datetime import date

# Local application imports
sys.path.append('..')
sys.path.append('../..')

from retrieve_product import *
from file_management import get_files_dir, check_save_file

# Get file directories
_, INPUT_DIR, OUTPUT_DIR = get_files_dir()

# colours
#1f9e89
#35b779
#80c066

/Users/elisamarquez/Documents/PhD/2semestre/Engineering_db/Engineering_db/src/03_Extract_information/01_Product_extraction/../retrieve_product.py:216: SyntaxWarning: invalid escape sequence '\S'
  """


<a class="anchor" id="Own_function_1"></a>
## <span class=title_1 style="color: #35b779">Own function</span>

In [2]:
def get_text(token_list):
    new_token_list = []
    if token_list:
        try:
            for token in token_list:
                    new_token_list.append(token.text)
            return(new_token_list)
        except:
            try:
                return([token_list.text])
            except:
                return(token_list)
        

In [3]:
def split_products(tokens):
    separated_products = []
    current_product = []
    doc_text = None
    
    for i, token in enumerate(tokens):
        # Get the doc text from the first token
        if doc_text is None and hasattr(token, 'doc'):
            doc_text = token.doc.text
            
        # If token ends with a comma, it should be treated as a separate product
        if token.text.endswith(','):
            current_product.append(token.text.strip(','))
            separated_products.append(" ".join(current_product))
            current_product = []
        else:
            # Check for "and" in the middle of two products
            if i > 0 and i < len(tokens) - 1:
                if tokens[i-1].text.endswith(',') and token.text.lower() == 'and' and tokens[i+1].text.isalpha():
                    separated_products.append(" ".join(current_product))
                    current_product = []
            current_product.append(token.text)

    # Add any remaining tokens as the last product
    if current_product:
        separated_products.append(" ".join(current_product))

    return [separated_products, doc_text]


<a class="anchor" id="Input_and_Output_files_2"></a>
# <span class=title_0 style="color: #1f9e89">Input and Output files</span>

<a class="anchor" id="Input_3"></a>
## <span class=title_1 style="color: #35b779">Input</span>

Articles that fit into the metabolic engineering classification

In [4]:
import pandas as pd

In [5]:
# Classify articles with full text
# file = OUTPUT_DIR + '/Articles/classified_articles_v_2025_06_30.csv'
file_path = '../00_Full_text_extraction/full_text_articles.json.gz'

# Define memory-efficient dtype mapping
dtype_spec = {
    'PM_ID': 'int32',      # PubMed IDs can be large, but int32 is sufficient for most cases
    'Year': 'int16',       # Years (2000-2025) easily fit in int16
    'PMC_ID': 'Int32',      # Use nullable integer type for PMC IDs (may contain missing values)
    'Title': 'string',
    'Abstract': 'string',
    'Journal': 'category', 
    'DOI': 'string',
    'Type': 'string',   
    'Author': 'string', 
    'Text': 'string'
}

# Load data with optimized memory usage
relevant_articles = pd.read_json(
    file_path,
    dtype=dtype_spec,
    orient='index',
    compression='gzip'
)

relevant_articles.shape

(16271, 9)

In [6]:
# Filter out review articles (case-insensitive check)
relevant_articles = relevant_articles.loc[
    ~relevant_articles['Type'].str.lower().str.contains('review', na=False)
]

relevant_articles.shape

(16271, 9)

In [7]:
relevant_articles.Text.dropna()

10662693    cm7204.qxd  03/03/2000  08:38  Page 77

Format...
10708651    Protein Engineering vol.13 no.2 pp.121–128, 20...
10742205    APPLIED AND ENVIRONMENTAL MICROBIOLOGY,
0099-2...
10742260    APPLIED AND ENVIRONMENTAL MICROBIOLOGY,
0099-2...
10835112    Protein Engineering vol.13 no.5 pp.377–384, 20...
                                  ...                        
40558924    Introduction:
The goal of our study was to con...
40559434    Introduction:
In the present work, the biosynt...
40564374    Introduction:
In this article, we report the e...
40572067    Introduction:
This study aims to enhance the e...
40572208    Introduction:
This article reports the first s...
Name: Text, Length: 8324, dtype: string

In [8]:
relevant_articles = relevant_articles.loc[
    ~relevant_articles['Abstract'].str.lower().str.contains('review', na=False)
]

In [9]:
relevant_articles.shape

(16130, 9)

<a class="anchor" id="Output_4"></a>
## <span class=title_1 style="color: #35b779">Output</span>

In [10]:
general_name = 'filtered_metabolic_eng_articles_with_raw_products'

today = date.today()
today = today.strftime("%Y_%m_%d")

output_file = f'{general_name}_V_{today}.pickle'
output_file

'filtered_metabolic_eng_articles_with_raw_products_V_2025_09_30.pickle'

<a class="anchor" id="Extract_product_5"></a>
# <span class=title_0 style="color: #1f9e89">Extract product</span>

Agregar que busque si el nombre es muy genérico en título (como biofuel)

<a class="anchor" id="From_title_6"></a>
## <span class=title_1 style="color: #35b779">From title</span>

In [11]:
testing = relevant_articles.head(60).copy()

In [12]:
testing.loc[:,'Product'] = testing.Title.apply(get_product_mlp_cache_track)

In [13]:
testing.dropna(subset='Product').head(60)

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",<NA>,[zeaxanthin]
10653745,A novel genetically engineered pathway for syn...,A new pathway to synthesize poly(hydroxyalkano...,Applied and environmental microbiology,2000,91890,10.1128/AEM.66.2.739-743.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:S J,LastName:Liu] [ForeName:A,LastNa...",<NA>,[poly(hydroxyalkanoic-acids)]
10689078,Metabolic engineering of Alcaligenes eutrophus...,The regulatory mechanisms of the biosynthesis ...,Enzyme and microbial technology,2000,<NA>,10.1016/s0141-0229(99)00156-8,"Journal Article,","[ForeName:Y,LastName:Jung] [ForeName:J,LastNam...",<NA>,[polyhydroxyalkanoate]
10742205,Properties of engineered poly-3-hydroxyalkanoa...,To prepare medium-chain-length poly-3-hydroxya...,Applied and environmental microbiology,2000,91986,10.1128/AEM.66.4.1311-1320.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:Q,LastName:Ren] [ForeName:N,LastName...","APPLIED AND ENVIRONMENTAL MICROBIOLOGY, 0099-2...",[poly-3-hydroxyalkanoates]
10802621,Improving lycopene production in Escherichia c...,Metabolic engineering has achieved encouraging...,Nature biotechnology,2000,<NA>,10.1038/75398,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:W R,LastName:Farmer] [ForeName:J C,L...",<NA>,[lycopene]
10814415,A short total synthesis of (+)-furanomycin.,[reaction: see text] Furanomycin is a Streptom...,Organic letters,2000,<NA>,10.1021/ol005569j,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:M P,LastName:VanBrunt] [ForeName:R F...",<NA>,[(+)-furanomycin]
10818234,Genetic evidence of branching in the isoprenoi...,An alternative mevalonate-independent pathway ...,FEBS letters,2000,<NA>,10.1016/s0014-5793(00)01552-0,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Rodríguez-Concepción] [Fo...",<NA>,"[isopentenyl, diphosphate, dimethylallyl, diph..."
10820335,Production of enantiopure styrene oxide by rec...,A whole cell biocatalytic process was develope...,Biotechnology and bioengineering,2000,<NA>,10.1002/(sici)1097-0290(20000705)69:1<91::aid-...,"Journal Article,","[ForeName:S,LastName:Panke] [ForeName:M G,Last...",<NA>,"[styrene, oxide]"
10849853,Polyhydroxybutyrate production from carbon dio...,Genetic characterization and enhancement of po...,Applied biochemistry and biotechnology,2000,<NA>,10.1385/abab:84-86:1-9:991,"Journal Article,","[ForeName:M,LastName:Miyake] [ForeName:K,LastN...",<NA>,[polyhydroxybutyrate]
10862673,Recombinant protein production driven by the t...,Batch processes for recombinant gene expressio...,Biotechnology and bioengineering,2000,<NA>,10.1002/1097-0290(20000820)69:4<351::aid-bit1>...,"Journal Article,","[ForeName:L,LastName:Chevalet] [ForeName:A,Las...",<NA>,[protein]


In [14]:
testing.dropna(subset='Product').head(100)

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",<NA>,[zeaxanthin]
10653745,A novel genetically engineered pathway for syn...,A new pathway to synthesize poly(hydroxyalkano...,Applied and environmental microbiology,2000,91890,10.1128/AEM.66.2.739-743.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:S J,LastName:Liu] [ForeName:A,LastNa...",<NA>,[poly(hydroxyalkanoic-acids)]
10689078,Metabolic engineering of Alcaligenes eutrophus...,The regulatory mechanisms of the biosynthesis ...,Enzyme and microbial technology,2000,<NA>,10.1016/s0141-0229(99)00156-8,"Journal Article,","[ForeName:Y,LastName:Jung] [ForeName:J,LastNam...",<NA>,[polyhydroxyalkanoate]
10742205,Properties of engineered poly-3-hydroxyalkanoa...,To prepare medium-chain-length poly-3-hydroxya...,Applied and environmental microbiology,2000,91986,10.1128/AEM.66.4.1311-1320.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:Q,LastName:Ren] [ForeName:N,LastName...","APPLIED AND ENVIRONMENTAL MICROBIOLOGY, 0099-2...",[poly-3-hydroxyalkanoates]
10802621,Improving lycopene production in Escherichia c...,Metabolic engineering has achieved encouraging...,Nature biotechnology,2000,<NA>,10.1038/75398,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:W R,LastName:Farmer] [ForeName:J C,L...",<NA>,[lycopene]
10814415,A short total synthesis of (+)-furanomycin.,[reaction: see text] Furanomycin is a Streptom...,Organic letters,2000,<NA>,10.1021/ol005569j,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:M P,LastName:VanBrunt] [ForeName:R F...",<NA>,[(+)-furanomycin]
10818234,Genetic evidence of branching in the isoprenoi...,An alternative mevalonate-independent pathway ...,FEBS letters,2000,<NA>,10.1016/s0014-5793(00)01552-0,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Rodríguez-Concepción] [Fo...",<NA>,"[isopentenyl, diphosphate, dimethylallyl, diph..."
10820335,Production of enantiopure styrene oxide by rec...,A whole cell biocatalytic process was develope...,Biotechnology and bioengineering,2000,<NA>,10.1002/(sici)1097-0290(20000705)69:1<91::aid-...,"Journal Article,","[ForeName:S,LastName:Panke] [ForeName:M G,Last...",<NA>,"[styrene, oxide]"
10849853,Polyhydroxybutyrate production from carbon dio...,Genetic characterization and enhancement of po...,Applied biochemistry and biotechnology,2000,<NA>,10.1385/abab:84-86:1-9:991,"Journal Article,","[ForeName:M,LastName:Miyake] [ForeName:K,LastN...",<NA>,[polyhydroxybutyrate]
10862673,Recombinant protein production driven by the t...,Batch processes for recombinant gene expressio...,Biotechnology and bioengineering,2000,<NA>,10.1002/1097-0290(20000820)69:4<351::aid-bit1>...,"Journal Article,","[ForeName:L,LastName:Chevalet] [ForeName:A,Las...",<NA>,[protein]


In [15]:
re_pat.sub('', testing.Title.iloc[1])

'Expression of Alcaligenes eutrophus flavohemoprotein and  Vitreoscilla  fusion protein for  hypoxic growth of Escherichia coli.'

In [16]:
'a'.endswith(tuple(['a','b']))

True

In [17]:
testing.Title.iloc[1]

'Expression of Alcaligenes eutrophus flavohemoprotein and engineered Vitreoscilla hemoglobin-reductase fusion protein for improved hypoxic growth of Escherichia coli.'

In [18]:
relevant_articles.loc[:,'Product'] = relevant_articles.Title.apply(get_product_mlp_cache_track)
title_product_count = len(relevant_articles['Product'].dropna())
from_title = relevant_articles.Product.dropna().index

--------CHECK REQUIRED--------
Result: acid, Extra: australian
Full context: acid australian brassica napus and B. juncea production
--------CHECK REQUIRED--------
Result: acid, Extra: australian
Full context: acid australian brassica napus and B. juncea production
--------CHECK REQUIRED--------
Result: cosmetic, Extra: plant
Full context: production of the food and cosmetic plant pigment bixin (annatto)
--------CHECK REQUIRED--------
Result: cosmetic, Extra: plant
Full context: production of the food and cosmetic plant pigment bixin (annatto)
--------CHECK REQUIRED--------
Result: saccharidic, Extra: moieties
Full context: production of conjugatable saccharidic moieties of gm2 and gm3 gangliosides
--------CHECK REQUIRED--------
Result: saccharidic, Extra: moieties
Full context: production of conjugatable saccharidic moieties of gm2 and gm3 gangliosides
--------CHECK REQUIRED--------
Result: antibiotic, Extra: inhibits
Full context: rhizobium etli usda9032 for the production of a phena

Result: antiarrhythmic, Extra: alkaloid
Full context: production of antiarrhythmic alkaloid ajmaline
--------CHECK REQUIRED--------
Result: antiarrhythmic, Extra: alkaloid
Full context: production of antiarrhythmic alkaloid ajmaline
--------CHECK REQUIRED--------
Result: characteristic, Extra: component
Full context: production of rhodiola rosea characteristic component rosavin
--------CHECK REQUIRED--------
Result: characteristic, Extra: component
Full context: production of rhodiola rosea characteristic component rosavin
--------CHECK REQUIRED--------
Result: acid, Extra: phosphatase
Full context: acid phosphatase production
--------CHECK REQUIRED--------
Result: acid, Extra: phosphatase
Full context: acid phosphatase production
--------CHECK REQUIRED--------
Result: polymeric, Extra: substance
Full context: polymeric substance synthesis
--------CHECK REQUIRED--------
Result: polymeric, Extra: substance
Full context: polymeric substance synthesis
--------CHECK REQUIRED--------
Result

In [19]:
relevant_articles['Product'].dropna()

10618204                                       [zeaxanthin]
10653745                      [poly(hydroxyalkanoic-acids)]
10689078                             [polyhydroxyalkanoate]
10742205                         [poly-3-hydroxyalkanoates]
10802621                                         [lycopene]
                                 ...                       
40562243                             [protocatechuic, acid]
40564374    [poly(3-hydroxybutyrate-co-3-hydroxyhexanoate)]
40567000                          [astaxanthin, zeaxanthin]
40572067                                     [amylosucrase]
40579636                                            [γ-pga]
Name: Product, Length: 8401, dtype: object

In [20]:
print(f"Found products in {title_product_count} articles from titles")


Found products in 8401 articles from titles


In [21]:
len(check_noun_cache)

7593

In [22]:
print_cache_stats()

Overall cache hit rate: 25.6%
check_noun_cache: 2575/10168 (25.3% hit rate)
product_cache: 2588/10017 (25.8% hit rate)


In [23]:
cache_hits

{'check_noun_cache': 2575, 'product_cache': 2588}

In [24]:
cache_misses

{'check_noun_cache': 7593, 'product_cache': 7429}

<a class="anchor" id="From_abstract_7"></a>
## <span class=title_1 style="color: #35b779">From abstract</span>

In [ ]:
articles_no_product_from_title = relevant_articles.loc[relevant_articles['Product'].isna()]
articles_no_product_from_title.loc[:, 'Product'] = articles_no_product_from_title['Abstract'].apply(get_product_mlp_cache_track)
abstract_product_count = len(articles_no_product_from_title['Product'].dropna())
from_abstract = articles_no_product_from_title.Product.dropna().index

In [26]:
print_cache_stats()

Overall cache hit rate: 13.6%
check_noun_cache: 3965/30514 (13.0% hit rate)
product_cache: 3941/27774 (14.2% hit rate)


In [27]:
print(f"Found products in {abstract_product_count} articles from abstracts")

Found products in 5094 articles from abstracts


In [28]:
relevant_articles.loc[from_abstract,'Product'] = articles_no_product_from_title.loc[:,'Product']

<a class="anchor" id="From_full_text_8"></a>
## <span class=title_1 style="color: #35b779">From full text</span>

In [ ]:
# Third pass: full text extraction for remaining articles
articles_still_missing_products = relevant_articles.loc[relevant_articles['Product'].isna()]
articles_still_missing_products.loc[:, 'Product'] = articles_still_missing_products['Text'].apply(
    lambda x: get_product_mlp_cache_track(x) if pd.notna(x) else x
)

full_text_product_count = len(articles_still_missing_products['Product'].dropna())
from_full_text = articles_still_missing_products.Product.dropna().index

In [30]:
print_cache_stats()

Overall cache hit rate: 13.0%
check_noun_cache: 4615/37256 (12.4% hit rate)
product_cache: 4533/33349 (13.6% hit rate)


In [31]:
print(f"Found products in {full_text_product_count} articles from full text")

Found products in 1431 articles from full text


In [32]:
relevant_articles.loc[from_full_text,'Product'] = articles_still_missing_products.loc[:,'Product']

In [33]:
# NEW: Add origin tracking column
relevant_articles['Product_Source'] = 'not_found'
relevant_articles.loc[from_title, 'Product_Source'] = 'title'
relevant_articles.loc[from_abstract, 'Product_Source'] = 'abstract'
relevant_articles.loc[from_full_text, 'Product_Source'] = 'full_text'

relevant_articles['Product_Source'] = relevant_articles['Product_Source'].astype('category')


In [34]:
relevant_articles.Product_Source.value_counts()

Product_Source
title        8401
abstract     5094
full_text    1431
not_found    1204
Name: count, dtype: int64

In [35]:
# Verify counts
print("Extraction results:")
print(f"From title: {title_product_count}")
print(f"From abstract: {abstract_product_count}") 
print(f"From full text: {full_text_product_count}")
print(f"Not found: {len(relevant_articles[relevant_articles['Product_Source'] == 'not_found'])}")

Extraction results:
From title: 8401
From abstract: 5094
From full text: 1431
Not found: 1204


<a class="anchor" id="Result_9"></a>
# <span class=title_0 style="color: #1f9e89">Result</span>

<a class="anchor" id="Statistics_10"></a>
## <span class=title_1 style="color: #35b779">Statistics </span>

In [36]:
from IPython.display import display, HTML
from display_texts import product_extraction_result, product_extraction_summary

In [37]:
total_articles = len(relevant_articles)
articles_with_products = len(relevant_articles['Product'].dropna())


In [38]:
result_text= product_extraction_result(total_articles,articles_with_products,title_product_count,abstract_product_count,full_text_product_count)

In [39]:
display(HTML(result_text))

<a class="anchor" id="Format_output_11"></a>
## <span class=title_1 style="color: #35b779">Format output</span>

To save into a pickle, we must format this...

In [40]:
mask = relevant_articles['Product'].notna()
result_df = (
    relevant_articles.loc[mask, 'Product'].apply(
        lambda x: split_products(x) 
    )
)

In [41]:
result_df

10618204         [[zeaxanthin], production of zeaxanthin and]
10618209          [[heme proteins], heme proteins production]
10649237                [[ethanol], ethanol production rates]
10649449                      [[sterol], sterol biosynthesis]
10653745    [[poly(hydroxyalkanoic-acids)], A genetically ...
                                  ...                        
40572067    [[amylosucrase], synergistic of the twin-argin...
40572208    [[nitrogenase], for the production of nitrogen...
40573728    [[cucurbitane-type mogrosides], the plant sira...
40577193    [[ethanol], the production of ethanol as one o...
40579636                          [[γ-pga], γ-pga production]
Name: Product, Length: 14926, dtype: object

In [42]:
# Create DataFrame from results and join
result_df = pd.DataFrame(result_df.to_list(), index=mask[mask].index, 
                        columns=['Separated_products', 'Doc_text'])
joined_data = relevant_articles.join(result_df)

In [43]:
joined_data.head()

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product,Product_Source,Separated_products,Doc_text
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",<NA>,[zeaxanthin],title,[zeaxanthin],production of zeaxanthin and
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",<NA>,"[heme, proteins]",abstract,[heme proteins],heme proteins production
10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,<NA>,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",<NA>,None,not_found,NaN,NaN
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,<NA>,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",<NA>,[ethanol],abstract,[ethanol],ethanol production rates
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,<NA>,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",<NA>,[sterol],abstract,[sterol],sterol biosynthesis


In [44]:
joined_data = joined_data.loc[:,~joined_data.columns.isin(['Product'])]
joined_data_raw = joined_data.explode('Separated_products')

In [45]:
joined_data_raw

,Title,Abstract,Journal,Year,PMC_ID,DOI,Type,Author,Text,Product_Source,Separated_products,Doc_text
10618204,Increased production of zeaxanthin and other p...,The psbAII locus was used as an integration pl...,Applied and environmental microbiology,2000,91786,10.1128/AEM.66.1.64-72.2000,"Journal Article,Research Support, U.S. Gov't, ...","[ForeName:D,LastName:Lagarde] [ForeName:L,Last...",<NA>,title,zeaxanthin,production of zeaxanthin and
10618209,Expression of Alcaligenes eutrophus flavohemop...,Expression of the vhb gene encoding hemoglobin...,Applied and environmental microbiology,2000,91791,10.1128/AEM.66.1.98-104.2000,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:A D,LastName:Frey] [ForeName:J E,Las...",<NA>,abstract,heme proteins,heme proteins production
10631776,Environmental biotechnology.,There is an increasing interest in environment...,Trends in biotechnology,2000,<NA>,10.1016/s0167-7799(99)01399-2,"Journal Article,","[ForeName:L P,LastName:Wackett]",<NA>,not_found,NaN,NaN
10649237,Altered regulation of pyruvate kinase or co-ov...,Glycolytic fluxes in resting Escherichia coli ...,Biotechnology and bioengineering,2000,<NA>,10.1002/(sici)1097-0290(20000305)67:5<623::aid...,"Journal Article,Research Support, Non-U.S. Gov't,","[ForeName:M,LastName:Emmerling] [ForeName:J E,...",<NA>,abstract,ethanol,ethanol production rates
10649449,Cloning and characterization of the Yarrowia l...,The squalene synthase (SQS) gene encodes a key...,"Yeast (Chichester, England)",2000,<NA>,10.1002/(SICI)1097-0061(200002)16:3<197::AID-Y...,"Journal Article,Research Support, Non-U.S. Gov...","[ForeName:S,LastName:Merkulov] [ForeName:F,Las...",<NA>,abstract,sterol,sterol biosynthesis
...,...,...,...,...,...,...,...,...,...,...,...,...
40572208,Heterologous Expression of the Nitrogen-Fixing...,Microbially mediated biological nitrogen fixat...,Microorganisms,2025,12195393,10.3390/microorganisms13061320,"Journal Article,","[ForeName:Xiuling,LastName:Wang] [ForeName:Shi...",Introduction: This article reports the first s...,abstract,nitrogenase,for the production of nitrogenase
40573728,Functional Characterization of Squalene Epoxid...,The medicinal plant <i>Siraitia grosvenorii</i...,"Plants (Basel, Switzerland)",2025,<NA>,10.3390/plants14121740,"Journal Article,","[ForeName:Huan,LastName:Zhao] [ForeName:Ze,Las...",<NA>,abstract,cucurbitane-type mogrosides,the plant siraitia grosvenorii the production ...
40577193,Structural characterization and dynamics of Ad...,"<i>Clostridium thermocellum</i>, a cellulolyti...",eLife,2025,<NA>,10.7554/eLife.96966,"Journal Article,","[ForeName:Samantha J,LastName:Ziegler] [ForeNa...",<NA>,abstract,ethanol,the production of ethanol as one of main
40578703,Overcoming glucose repression through cellobio...,Pectin-rich biomass is a promising substrate f...,Bioresource technology,2025,<NA>,10.1016/j.biortech.2025.132892,"Journal Article,","[ForeName:Dahye,LastName:Lee] [ForeName:Deokye...",<NA>,not_found,NaN,NaN


In [46]:
output_file

'filtered_metabolic_eng_articles_with_raw_products_V_2025_09_30.pickle'

In [47]:
joined_data_raw.to_pickle(output_file)

<a class="anchor" id="Summary_12"></a>
# <span class=title_0 style="color: #1f9e89">Summary</span>

In [48]:
summary_text = product_extraction_summary(total_articles,articles_with_products,title_product_count,abstract_product_count,full_text_product_count, output_file)

In [49]:
joined_data_raw.to_pickle(output_file)

In [50]:
display(HTML(summary_text))